# **Import Library**

In [14]:
import gdown
import pandas as pd
import os
import numpy as np

# **Proses Pengolahan File Dietary Interview**

In [15]:
url1 = "https://drive.google.com/uc?export=download&id=1C555bMMGOXjpsPb79eH7QRJ7VE4fZOKn"
url2 = "https://drive.google.com/uc?export=download&id=1gjarsQakV4tNULtULnj_P-KKgfWIAJ_x"

In [16]:
# ============================================================
# 2. MEMBACA KEMBALI FILE CSV
# ============================================================

gdown.download(
    url1,
    "DR1IFF_J.csv",
    quiet=False,
    fuzzy=True
)

gdown.download(
    url2,
    "DR2IFF_J.csv",
    quiet=False,
    fuzzy=True
)

df_day1 = pd.read_csv("DR1IFF_J.csv")
df_day2 = pd.read_csv("DR2IFF_J.csv")

print("\n==========================================")
print("DATA CSV")
print("==========================================")

print("Ukuran Day 1:", df_day1.shape)
print("Ukuran Day 2:", df_day2.shape)

Downloading...
From (original): https://drive.google.com/uc?id=1C555bMMGOXjpsPb79eH7QRJ7VE4fZOKn
From (redirected): https://drive.google.com/uc?id=1C555bMMGOXjpsPb79eH7QRJ7VE4fZOKn&confirm=t&uuid=8a18db24-2d9d-4354-b2ae-7394aa18ce2d
To: /content/DR1IFF_J.csv
100%|██████████| 109M/109M [00:00<00:00, 130MB/s]
Downloading...
From: https://drive.google.com/uc?id=1gjarsQakV4tNULtULnj_P-KKgfWIAJ_x
To: /content/DR2IFF_J.csv
100%|██████████| 90.7M/90.7M [00:00<00:00, 179MB/s]



DATA CSV
Ukuran Day 1: (112683, 84)
Ukuran Day 2: (93500, 84)


In [17]:
# ============================================================
# 3. CEK KOLOM YANG DIPERLUKAN
# ============================================================

print("\nKolom Day 1:")
print(
    df_day1[
        [
            "SEQN",
            "DR1IFDCD",
            "DR1IGRMS"
        ]
    ].head()
)

print("\nKolom Day 2:")
print(
    df_day2[
        [
            "SEQN",
            "DR2IFDCD",
            "DR2IGRMS"
        ]
    ].head()
)


Kolom Day 1:
      SEQN    DR1IFDCD  DR1IGRMS
0  93704.0  55100050.0     80.58
1  93704.0  11513600.0    248.00
2  93704.0  24198739.0    100.00
3  93704.0  74401010.0      5.67
4  93704.0  71401020.0    112.00

Kolom Day 2:
      SEQN    DR2IFDCD  DR2IGRMS
0  93704.0  64104010.0    155.00
1  93704.0  57241000.0     27.75
2  93704.0  11112210.0    122.00
3  93704.0  63149010.0     77.50
4  93704.0  52302010.0    130.00


In [18]:
# ============================================================
# 4. MENYAMAKAN NAMA KOLOM DAY 2
# ============================================================

df_day2 = df_day2.rename(
    columns={
        "DR2IFDCD": "DR1IFDCD",
        "DR2IGRMS": "DR1IGRMS"
    }
)

In [19]:
# ============================================================
# 5. MENAMBAHKAN IDENTITAS HARI
# ============================================================

df_day1["DAY"] = 1
df_day2["DAY"] = 2

In [20]:
# ============================================================
# 6. MENGGABUNGKAN DAY 1 DAN DAY 2
# ============================================================

df_combined = pd.concat(
    [
        df_day1[
            [
                "SEQN",
                "DR1IFDCD",
                "DR1IGRMS",
                "DAY"
            ]
        ],

        df_day2[
            [
                "SEQN",
                "DR1IFDCD",
                "DR1IGRMS",
                "DAY"
            ]
        ]
    ],
    ignore_index=True
)

print("\n==========================================")
print("DATA GABUNGAN")
print("==========================================")

print("Ukuran data gabungan:")
print(df_combined.shape)


DATA GABUNGAN
Ukuran data gabungan:
(206183, 4)


In [21]:
# ============================================================
# 7. CEK JUMLAH RESPONDEN PER HARI
# ============================================================

jumlah_responden_per_hari = (
    df_combined
    .groupby("DAY")["SEQN"]
    .nunique()
)

print("\nJumlah responden per hari:")
print(jumlah_responden_per_hari)


Jumlah responden per hari:
DAY
1    7640
2    6639
Name: SEQN, dtype: int64


In [22]:
# ============================================================
# 8. MENENTUKAN RESPONDEN YANG MEMILIKI 2 HARI
# ============================================================

jumlah_hari = (
    df_combined
    .groupby("SEQN")["DAY"]
    .nunique()
)

print("\nDistribusi jumlah hari:")
print(
    jumlah_hari
    .value_counts()
    .sort_index()
)


Distribusi jumlah hari:
DAY
1    1003
2    6638
Name: count, dtype: int64


In [23]:
# ============================================================
# 9. MENGAMBIL SEQN YANG MEMILIKI DAY 1 DAN DAY 2
# ============================================================

seqn_2hari = jumlah_hari[
    jumlah_hari == 2
].index


# Filter dataframe
df_combined_2hari = df_combined[
    df_combined["SEQN"].isin(seqn_2hari)
].copy()


print("\nJumlah responden dengan 2 hari:")
print(
    df_combined_2hari["SEQN"].nunique()
)


Jumlah responden dengan 2 hari:
6638


In [24]:
# ============================================================
# 10. VALIDASI DATA 2 HARI
# ============================================================

validasi_hari = (
    df_combined_2hari
    .groupby("SEQN")["DAY"]
    .nunique()
)

print(
    "\nApakah seluruh responden memiliki 2 hari?",
    (validasi_hari == 2).all()
)


Apakah seluruh responden memiliki 2 hari? True


In [25]:
# ============================================================
# 11. KONVERSI TIPE DATA
# ============================================================

df_combined_2hari["DR1IFDCD"] = pd.to_numeric(
    df_combined_2hari["DR1IFDCD"],
    errors="coerce"
)

df_combined_2hari["DR1IGRMS"] = pd.to_numeric(
    df_combined_2hari["DR1IGRMS"],
    errors="coerce"
)

In [26]:
# ============================================================
# 12. MEMBUAT KODE FOOD GROUP
# ============================================================

df_combined_2hari["food_group_code"] = (
    df_combined_2hari["DR1IFDCD"]
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
    .str[:2]
)

In [27]:
# ============================================================
# 13. MENGHAPUS NONCARBONATED WATER
#     KODE FNDDS 94
# ============================================================

jumlah_sebelum_94 = len(
    df_combined_2hari
)

df_combined_2hari = df_combined_2hari[
    df_combined_2hari["food_group_code"] != "94"
].copy()

jumlah_sesudah_94 = len(
    df_combined_2hari
)

print("\nJumlah baris sebelum menghapus kode 94:")
print(jumlah_sebelum_94)

print("\nJumlah baris setelah menghapus kode 94:")
print(jumlah_sesudah_94)

print("\nJumlah baris yang dihapus:")
print(
    jumlah_sebelum_94 - jumlah_sesudah_94
)


Jumlah baris sebelum menghapus kode 94:
192203

Jumlah baris setelah menghapus kode 94:
167686

Jumlah baris yang dihapus:
24517


In [28]:
# ============================================================
# 14. MENGAMBIL DIGIT PERTAMA
# ============================================================

df_combined_2hari["food_group_digit"] = (
    df_combined_2hari["food_group_code"]
    .str[0]
)

In [29]:
# ============================================================
# 15. MAPPING KELOMPOK PANGAN
# ============================================================

group_map = {

    "1": "milk_dairy",

    "2": "meat_poultry_fish",

    "3": "eggs",

    "4": "legumes_nuts_seeds",

    "5": "grain_products",

    "6": "fruits_and_juices",

    "7": "vegetables",

    "8": "fats_oils_dressings",

    "9": "sugars_sweets_beverages"
}


df_combined_2hari["food_group"] = (
    df_combined_2hari["food_group_digit"]
    .map(group_map)
)

In [30]:
# ============================================================
# 16. MEMBERSIHKAN DATA
# ============================================================

df_combined_2hari = (
    df_combined_2hari
    .dropna(
        subset=[
            "SEQN",
            "food_group",
            "DR1IGRMS"
        ]
    )
    .copy()
)


In [31]:
# ============================================================
# 17. MENGHAPUS NILAI GRAM NEGATIF
# ============================================================

df_combined_2hari = df_combined_2hari[
    df_combined_2hari["DR1IGRMS"] >= 0
].copy()

In [32]:
# ============================================================
# 18. DAFTAR 9 KELOMPOK PANGAN
# ============================================================

group_columns = [

    "milk_dairy",

    "meat_poultry_fish",

    "eggs",

    "legumes_nuts_seeds",

    "grain_products",

    "fruits_and_juices",

    "vegetables",

    "fats_oils_dressings",

    "sugars_sweets_beverages"
]

In [33]:
# ============================================================
# 19. MENJUMLAHKAN KONSUMSI
#     BERDASARKAN SEQN + DAY + FOOD GROUP
# ============================================================

food_daily = (
    df_combined_2hari
    .groupby(
        [
            "SEQN",
            "DAY",
            "food_group"
        ],
        as_index=False
    )["DR1IGRMS"]
    .sum()
)

In [34]:
# ============================================================
# 20. PIVOT FOOD GROUP
# ============================================================

food_daily_pivot = (
    food_daily
    .pivot_table(
        index=[
            "SEQN",
            "DAY"
        ],
        columns="food_group",
        values="DR1IGRMS",
        aggfunc="sum",
        fill_value=0
    )
)


food_daily_pivot.columns.name = None

food_daily_pivot = (
    food_daily_pivot
    .reset_index()
)


In [35]:
# ============================================================
# 21. MEMASTIKAN SEMUA KELOMPOK TERSEDIA
# ============================================================

for col in group_columns:

    if col not in food_daily_pivot.columns:

        food_daily_pivot[col] = 0

In [36]:
# ============================================================
# 22. MENGHITUNG RATA-RATA DAY 1 DAN DAY 2
# ============================================================

food_2hari = (
    food_daily_pivot
    .groupby("SEQN")[group_columns]
    .mean()
    .reset_index()
)

In [37]:
# ============================================================
# 23. MENGHITUNG TOTAL MAKANAN
# ============================================================

food_2hari["total_food_gram"] = (
    food_2hari[group_columns]
    .sum(axis=1)
)

In [38]:
# ============================================================
# 24. MENAMPILKAN HASIL
# ============================================================

print("\n==========================================")
print("HASIL AKHIR")
print("==========================================")

print(
    "Jumlah responden:",
    food_2hari["SEQN"].nunique()
)

print(
    "Jumlah food groups:",
    len(group_columns)
)

print(
    "Ukuran dataframe:",
    food_2hari.shape
)

display(
    food_2hari.head()
)


HASIL AKHIR
Jumlah responden: 6598
Jumlah food groups: 9
Ukuran dataframe: (6598, 11)


,SEQN,milk_dairy,meat_poultry_fish,eggs,legumes_nuts_seeds,grain_products,fruits_and_juices,vegetables,fats_oils_dressings,sugars_sweets_beverages,total_food_gram
0,93704.0,323.175,67.000,0.000,0.00,202.615,327.950,64.460,0.000,17.070,1002.270
1,93705.0,29.030,0.000,55.000,207.04,208.095,0.000,150.565,7.350,831.500,1488.580
2,93707.0,75.000,231.000,105.000,0.00,108.000,0.000,93.230,12.835,133.000,758.065
3,93708.0,138.065,176.455,0.000,18.25,212.940,37.500,118.125,8.775,34.000,744.110
4,93710.0,492.200,28.500,12.555,1.75,190.175,171.695,26.625,0.000,3.335,926.835


In [39]:
# ============================================================
# 25. STATISTIK DESKRIPTIF
# ============================================================

print("\n==========================================")
print("STATISTIK DESKRIPTIF")
print("==========================================")

display(
    food_2hari[group_columns].describe()
)


STATISTIK DESKRIPTIF


,milk_dairy,meat_poultry_fish,eggs,legumes_nuts_seeds,grain_products,fruits_and_juices,vegetables,fats_oils_dressings,sugars_sweets_beverages
count,6598.000000,6598.000000,6598.000000,6598.000000,6598.000000,6598.000000,6598.000000,6598.000000,6598.000000
mean,223.130149,178.357233,24.407761,31.563543,304.286967,159.708913,135.930376,13.058860,659.089168
std,249.179836,157.692516,42.550179,72.111921,223.144100,197.441478,150.828842,20.537381,717.478829
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,37.500000,67.000000,0.000000,0.000000,148.500000,0.000000,31.598750,0.000000,186.000000
50%,150.062500,143.340000,0.000000,0.000000,262.000000,103.000000,90.577500,4.665000,481.985000
75%,321.620000,248.606250,45.500000,34.690000,409.468750,239.500000,189.348750,18.665000,913.545000
max,2528.000000,1494.750000,414.690000,1155.490000,2445.445000,2751.125000,1477.680000,235.000000,11700.000000


In [40]:
# ============================================================
# 26. MENYIMPAN HASIL FOOD GROUP
# ============================================================

food_2hari.to_csv(
    "NHANES_food_group_2day_average.csv",
    index=False
)

print(
    "\nFile berhasil disimpan:"
)

print(
    "NHANES_food_group_2day_average.csv"
)


File berhasil disimpan:
NHANES_food_group_2day_average.csv


In [41]:

# ============================================================
# 27. AUDIT SUB-KATEGORI KELOMPOK 9 (N = 2.324)
# ============================================================

# Baca dataset akhir
df_final = pd.read_csv("NHANES_food_group_2day_average.csv")
valid_seqn = df_final["SEQN"].unique()

# Ambil data Day 1 dan Day 2
d1 = df_day1.loc[
    df_day1["SEQN"].isin(valid_seqn),
    ["SEQN", "DR1IFDCD", "DR1IGRMS", "DR1ISUGR"]
].copy()

d2 = df_day2.loc[
    df_day2["SEQN"].isin(valid_seqn),
    ["SEQN", "DR1IFDCD", "DR1IGRMS", "DR2ISUGR"]
].copy()

# Samakan nama kolom dan gabungkan
d1.columns = d2.columns = ["SEQN", "CODE", "GRAMS", "SUGAR"]
items = pd.concat([d1, d2], ignore_index=True)

# Filter Kelompok 9 dan keluarkan air putih (kode 94)
items["SUB2"] = items["CODE"].astype("Int64").astype(str).str[:2]
g9 = items[items["SUB2"].str.startswith("9") & (items["SUB2"] != "94")].copy()

# Hitung total dan persentase setiap subkategori
overall = g9.groupby("SUB2").agg(
    total_grams=("GRAMS", "sum"),
    total_sugar=("SUGAR", "sum")
)

overall["pct_grams"] = (overall["total_grams"] / overall["total_grams"].sum() * 100).round(2)
overall["pct_sugar"] = (overall["total_sugar"] / overall["total_sugar"].sum() * 100).round(2)

print("=== AUDIT KELOMPOK 9 (N = 2.324) ===")
display(overall)

# Analisis berdasarkan cluster jika tersedia
if "Cluster" in df_final.columns:
    g9_clust = g9.merge(df_final[["SEQN", "Cluster"]], on="SEQN")
    by_clust = g9_clust.groupby(["Cluster", "SUB2"])["GRAMS"].sum().unstack(fill_value=0)
    print("\n=== PROPORSI BERAT SUB-KODE PER KLASTER (%) ===")
    display((by_clust.div(by_clust.sum(axis=1), axis=0) * 100).round(2))

=== AUDIT KELOMPOK 9 (N = 2.324) ===


,total_grams,total_sugar,pct_grams,pct_sugar
SUB2,,,,
91,210960.53,115603.44,2.43,24.98
92,7008808.34,305794.37,80.59,66.08
93,997593.70,12800.02,11.47,2.77
95,479305.06,28599.69,5.51,6.18
